# k-Nearest Neighbours, Naive Bayes and Support Vector Machines – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. k-nearest neighbours (k-NN)
2. Scaling of features
3. Naive Bayes (worked example: play golf?)
4. Naive Bayes for text
5. Data leakage
6. Support vector machines (SVM)

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. k-nearest neighbours (k-NN)

1. Choose $k$.
2. Compute the distances of the new point $x$ to all training points.
3. Take the $k$ nearest: $\mathcal N_k(x)$.
4. Predict the majority class (classification) or the mean of $y$ (regression).

**Distance** (Euclidean – straight-line distance, as with a ruler):

$$d(x, x') = \sqrt{\sum_{j=1}^{p}(x_j - x'_j)^2}$$

**Class probability** = share of the $k$ nearest neighbours that belong to class $c$:
$\hat P(y = c \mid x) = \frac{1}{k}\sum_{i \in \mathcal N_k(x)} 1(y_i = c)$. Example: $k = 5$, three neighbours blue →
P(blue) = 3/5 = 0.6.

**Choice of $k$:** small $k$ → flexible, wiggly boundary (high variance); large $k$ → smooth boundary (high bias).
Choose $k$ by cross-validation. k-NN has no real training phase ("lazy learner") – all the work happens at prediction
time.

In [ ]:
# Simulate 250 points of two interleaving half moons
X, y = make_moons(n_samples=250, noise=0.35, random_state=0)

# Figure showing the decision regions of k-NN for k = 1 and k = 15
for k in [1, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    DecisionBoundaryDisplay.from_estimator(knn, X, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y)
    plt.title('k = ' + str(k))
    plt.show()

## 2. Scaling of features

**Caution:** scale the features when their units are not comparable – otherwise the variable with the largest scale
dominates the distance. Standardisation: $z_j = (x_j - \bar x_j)/s_j$.

Example: two persons differ by 30 years in age and by 300 CHF in income. Without scaling, the income difference
dominates completely; after standardising (e.g. $s_{\text{age}} = 15$ years, $s_{\text{income}} = 2'000$ CHF), the age
difference (2 standard deviations) matters much more than the income difference (0.15 standard deviations).

In scikit-learn the scaler belongs into a **pipeline** (`make_pipeline(StandardScaler(), KNeighborsClassifier())`), so
that it is fitted on the training data only. The same applies to SVMs.

In [ ]:
# Two persons differ by 30 years in age and by 300 CHF in income
diff = np.array([30, 300])
sd = np.array([15, 2000])  # standard deviations of age and income

# Euclidean distance without and with standardisation
print(
    'Distance unscaled:',
    round(np.sqrt(np.sum(diff**2)), 1),
    ' (dominated by income)',
)
print(
    'Distance standardised:',
    round(np.sqrt(np.sum((diff / sd) ** 2)), 2),
    ' (dominated by age)',
)

## 3. Naive Bayes

**Bayes' theorem** (Thomas Bayes, 1701–1761):

$$P(y \mid x) = \frac{P(x \mid y)\,P(y)}{P(x)}$$

In words: how likely is class $y$ after seeing $x$? Start from the **base rate** $P(y)$ and update it by how typical $x$
is for $y$.

**"Naive": the features are assumed to be independent given the class:**

$$P(y \mid x_1, \dots, x_p) \;\propto\; P(y)\prod_{j=1}^{p} P(x_j \mid y)$$

In words: multiply the base rate with one probability per feature; predict the class with the highest score.

**Worked example: play golf?** New day: outlook = sunny, temperature = cool, humidity = high, windy = true.

In [ ]:
# Naive Bayes by hand: play golf? (14 days)
golf = pd.DataFrame(
    [
        ['sunny', 'hot', 'high', False, 'no'],
        ['sunny', 'hot', 'high', True, 'no'],
        ['overcast', 'hot', 'high', False, 'yes'],
        ['rainy', 'mild', 'high', False, 'yes'],
        ['rainy', 'cool', 'normal', False, 'yes'],
        ['rainy', 'cool', 'normal', True, 'no'],
        ['overcast', 'cool', 'normal', True, 'yes'],
        ['sunny', 'mild', 'high', False, 'no'],
        ['sunny', 'cool', 'normal', False, 'yes'],
        ['rainy', 'mild', 'normal', False, 'yes'],
        ['sunny', 'mild', 'normal', True, 'yes'],
        ['overcast', 'mild', 'high', True, 'yes'],
        ['overcast', 'hot', 'normal', False, 'yes'],
        ['rainy', 'mild', 'high', True, 'no'],
    ],
    columns=['outlook', 'temp', 'humidity', 'windy', 'play'],
)
print(golf)

# Days with play = yes and with play = no
yes = golf[golf['play'] == 'yes']
no = golf[golf['play'] == 'no']

# New day: outlook = sunny, temp = cool, humidity = high, windy = True
# Score = base rate * share of each feature value among the days of the class
score_yes = (
    len(yes)
    / len(golf)
    * np.mean(yes['outlook'] == 'sunny')
    * np.mean(yes['temp'] == 'cool')
    * np.mean(yes['humidity'] == 'high')
    * np.mean(yes['windy'])
)
score_no = (
    len(no)
    / len(golf)
    * np.mean(no['outlook'] == 'sunny')
    * np.mean(no['temp'] == 'cool')
    * np.mean(no['humidity'] == 'high')
    * np.mean(no['windy'])
)
print('score(yes) =', round(score_yes, 4))
print('score(no)  =', round(score_no, 4))
print('P(no | x) =', round(score_no / (score_no + score_yes), 3))

**Result:** score(no) = 0.0206 > score(yes) = 0.0053 → predict "no". Normalised: P(no | x) = 0.0206 / (0.0206 + 0.0053)
≈ 0.795.

## 4. Naive Bayes for text

Texts are turned into word counts (**bag of words**): each column is a word of the vocabulary, each row a document; word
order is ignored. `MultinomialNB` estimates $P(\text{word} \mid y)$ from the word counts per class (with Laplace
smoothing) and multiplies them for all words of a document.

The ratio $\log P(\text{word} \mid \text{class } 1) - \log P(\text{word} \mid \text{class } 0)$ shows which words are
typical for each class.

In [ ]:
# Naive Bayes for text: six messages (1 = spam, 0 = normal message)
docs = [
    'free prize call now',
    'win a free prize',
    'call me later',
    'lunch tomorrow',
    'are you free for lunch',
    'claim your free cash prize',
]
labels = [1, 1, 0, 0, 0, 1]

# Bag of words: one column per word, one row per message
vec = CountVectorizer()
W = vec.fit_transform(docs)
words = vec.get_feature_names_out()
print(pd.DataFrame(W.toarray(), columns=words, index=docs))

# Fit Naive Bayes
nb = MultinomialNB()
nb.fit(W, labels)

# Log ratio: positive = typical for spam, negative = typical for normal
log_ratio = pd.Series(
    nb.feature_log_prob_[1] - nb.feature_log_prob_[0], index=words
)
print(log_ratio.sort_values(ascending=False).round(2))

# Probability of spam for a new message
new_message = vec.transform(['free lunch'])
print('P(spam | "free lunch") =', round(nb.predict_proba(new_message)[0, 1], 3))

## 5. Data leakage

**Data leakage** means that a model uses information that would **not be available at the time of the prediction** –
often information that is created *after* the outcome or that contains the target itself (e.g. a text field describing
the outcome, a variable measured after the event, or preprocessing fitted on the test data). The model then looks
excellent in the evaluation but fails in practice.

Checklist: for every feature ask *"Do I know this value at the moment I want to make the prediction?"* – and fit all
preprocessing steps (scaling, vocabulary of a text vectorizer, imputation) on the training data only.

## 6. Support vector machines (SVM)

- **Widest gap:** find the line (hyperplane $w^\top x + b = 0$) that separates the classes with the **largest margin**.
- **Support vectors:** the points closest to the line determine it – all others do not matter.
- **$C$ (soft margin):** with a soft margin, support vectors may lie inside the gap or even be misclassified. Small $C$:
  wide gap, some errors allowed; large $C$: few errors, narrow gap (risk of overfitting).
- **Kernel:** measures similarity in a transformed space → curved boundaries (kernel trick), e.g. the RBF kernel
  $K(x, x') = \exp(-\gamma\lVert x - x'\rVert^2)$.

Formally, the soft-margin SVM minimises hinge loss + penalty:
$\frac{1}{2}\lVert w\rVert^2 + C\sum_i \max\big(0, 1 - y_i(w^\top x_i + b)\big)$.

Interactive demo: jgreitemann.github.io/svm-demo

In [ ]:
# Function: draws the boundary (solid), the margin (dashed), the points and
# the circled support vectors of an SVM with two features
def plot_svm(svm, X, y, title):
    """Plot a fitted SVM with two features."""
    DecisionBoundaryDisplay.from_estimator(
        svm,
        X,
        response_method='decision_function',
        plot_method='contour',
        levels=[-1, 0, 1],
        linestyles=['--', '-', '--'],
        colors='black',
    )
    plt.scatter(X[:, 0], X[:, 1], c=y)
    sv = svm.support_vectors_
    plt.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none', edgecolors='red')
    plt.title(title + ': ' + str(len(sv)) + ' support vectors')
    plt.show()


# Linear SVM on two separable groups
X_lin, y_lin = make_blobs(
    n_samples=60, centers=[[-2, -1], [2, 1.5]], cluster_std=0.9, random_state=3
)
svm_lin = SVC(kernel='linear', C=1)
svm_lin.fit(X_lin, y_lin)
plot_svm(svm_lin, X_lin, y_lin, 'Linear SVM')

# SVM with RBF kernel on two circles (non-linear boundary)
X_circ, y_circ = make_circles(
    n_samples=200, noise=0.08, factor=0.4, random_state=0
)
svm_rbf = SVC(kernel='rbf', C=1)
svm_rbf.fit(X_circ, y_circ)
plot_svm(svm_rbf, X_circ, y_circ, 'RBF kernel')

In [ ]:
# Effect of C on the margin and the number of support vectors
# (linear kernel, overlapping classes)
X_ov, y_ov = make_blobs(
    n_samples=100,
    centers=[[-1, -0.5], [1, 0.5]],
    cluster_std=1.0,
    random_state=1,
)

# Figures showing the margin for C = 0.01, 1 and 100
for C in [0.01, 1, 100]:
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_ov, y_ov)
    plot_svm(svm, X_ov, y_ov, 'C = ' + str(C))

**Reading the plots:** the solid line is the decision boundary, the dashed lines the margin. Only the circled support
vectors determine the boundary. A small $C$ gives a wide margin with many support vectors (more bias), a large $C$ a
narrow margin with few support vectors (more variance).

**Key messages**
- k-NN and SVM are based on distances – **scale the features**.
- Naive Bayes multiplies the base rate with one probability per feature; it is simple, fast and works well for text.
- An SVM is determined by its support vectors; $C$ and the kernel control the flexibility.
- Always check whether a feature would be available at prediction time (data leakage).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')